# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

**Périmètre demandé : tests statistiques et segmentation uniquement. Le dashboard (section 3) reste à faire.**


In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('J4_Tests_Segmentation')
         .master('local[2]').config('spark.sql.shuffle.partitions', '8').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
ROOT = Path('/workspace')
REPORT = {'scope': 'Tests statistiques et segmentation ; dashboard non réalisé', 'seed': 42}
SILVER_MAIN_PATH = '/lakehouse/silver/flights'
silver = spark.read.format('delta').load(SILVER_MAIN_PATH)
print('Silver :', silver.count(), 'vols')

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7e158b3f-ef92-42e4-9c7e-7f1aab91497f;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central


	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central


	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 756ms :: artifacts dl 24ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jac

:: retrieving :: org.apache.spark#spark-submit-parent-7e158b3f-ef92-42e4-9c7e-7f1aab91497f
	confs: [default]
	0 artifacts copied, 33 already retrieved (0kB/10ms)
26/11/19 10:38:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Silver : 1943809 vols


## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

### Hypothèses et méthode

**Test 1 — Mann–Whitney bilatéral :** H0 : la distribution des retards au départ est la même le matin (06–11 h) et le soir (18–23 h). H1 : les distributions diffèrent. Ce test de rangs convient à une distribution très asymétrique et corrige les ex æquo en mode asymptotique. Il ne teste pas exclusivement la médiane et ne démontre pas que l'heure cause les retards.

**Test 2 — Kruskal–Wallis :** H0 : les quatre groupes mensuels ont la même distribution de rangs ; une interprétation en termes de position/médiane exige des formes comparables. H1 : au moins un groupe diffère. Le test global n'identifie pas les paires différentes : pas de conclusion post hoc sans test supplémentaire.

On retient un échantillon déterministe de **80 000 vols au maximum**, triés par hash d'une clé de vol, parmi les vols non annulés avec retard au départ renseigné. Seulement quatre colonnes utiles sont transférées vers Pandas. Ce sous-ensemble réduit la charge du driver ; il ne rend pas les observations indépendantes. Les p-values sont nominales sous cette hypothèse : des vols partagent dates, aéroports et perturbations. Correction de Holm sur nos deux tests, seuil familial 5 %, tailles d'effet et contrôle complémentaire par bootstrap des jours pour la comparaison matin/soir.

Les quatre mois représentent seulement l'extrait de 2015. Les horaires sont locaux. Aucun résultat ne décrit une année complète ni un effet causal.

In [2]:
observations = silver.filter((F.col('cancelled') == 0) & F.col('departure_delay').isNotNull())
KEY = ['flight_date', 'airline', 'flight_number', 'origin_airport', 'destination_airport', 'scheduled_departure']
# Sel fixe : sélection reproductible indépendamment de l'ordre des partitions.
sample = (observations.withColumn('_sample_hash', F.xxhash64(F.lit(42), *KEY))
          .orderBy('_sample_hash', *KEY).limit(80000)
          .select('flight_date', 'month', 'departure_slot', 'departure_delay'))
df_sample = sample.toPandas()
assert 0 < len(df_sample) <= 80000 and df_sample['departure_delay'].notna().all()
morning = df_sample.loc[df_sample.departure_slot == 'Matin', 'departure_delay'].to_numpy(dtype=float)
evening = df_sample.loc[df_sample.departure_slot == 'Soir', 'departure_delay'].to_numpy(dtype=float)
assert min(len(morning), len(evening)) > 100
mw = stats.mannwhitneyu(evening, morning, alternative='two-sided', method='asymptotic')
months = [1, 4, 7, 11]
groups = [df_sample.loc[df_sample.month == m, 'departure_delay'].to_numpy(dtype=float) for m in months]
assert min(map(len, groups)) >= 5
kw = stats.kruskal(*groups, nan_policy='raise')
# Holm : maximum cumulatif de (nombre d'hypothèses restantes) × p, dans l'ordre des p-values.
pvalues = np.array([mw.pvalue, kw.pvalue], dtype=float)
order = np.argsort(pvalues)
adjusted = np.empty(2)
adjusted[order] = np.minimum(1, np.maximum.accumulate(pvalues[order] * np.array([2, 1])))
REPORT['sampling'] = {'population_observed': observations.count(), 'sample_rows': len(df_sample),
                       'monthly_rows': {str(m): len(g) for m, g in zip(months, groups)}}
REPORT['tests'] = {
 'mann_whitney': {'h0': 'Distributions matin et soir identiques', 'alternative': 'two-sided',
   'n_morning': len(morning), 'n_evening': len(evening), 'statistic_U_evening': float(mw.statistic),
   'pvalue': float(mw.pvalue), 'pvalue_holm': float(adjusted[0]),
   'cliffs_delta_evening_vs_morning': float(2 * mw.statistic / (len(evening) * len(morning)) - 1),
   'median_morning': float(np.median(morning)), 'median_evening': float(np.median(evening)),
   'reject_h0_nominal': bool(adjusted[0] < .05)},
 'kruskal_wallis': {'h0': 'Distributions de rangs des quatre mois identiques',
   'statistic_H': float(kw.statistic), 'pvalue': float(kw.pvalue), 'pvalue_holm': float(adjusted[1]),
   'epsilon_squared': float(max(0, (kw.statistic - 4 + 1) / (sum(map(len, groups)) - 4))),
   'monthly_medians': {str(m): float(np.median(g)) for m, g in zip(months, groups)},
   'reject_h0_nominal': bool(adjusted[1] < .05)} }
# Sensibilité : agréger tous les vols matin/soir par jour, puis rééchantillonner les 122 jours.
# Une paire de moyennes par jour ; chaque jour a le même poids. Ne confondre avec l'effet de rang du test U.
daily = (observations.filter(F.col('departure_slot').isin('Matin', 'Soir'))
         .groupBy('flight_date').pivot('departure_slot', ['Matin', 'Soir']).agg(F.avg('departure_delay'))
         .orderBy('flight_date').toPandas().dropna())
diffs = (daily['Soir'] - daily['Matin']).to_numpy(dtype=float)
rng = np.random.default_rng(42)
boot = diffs[rng.integers(0, len(diffs), size=(5000, len(diffs)))].mean(axis=1)
REPORT['day_bootstrap'] = {'days': len(diffs), 'replicates': 5000,
 'mean_daily_evening_minus_morning_minutes': float(diffs.mean()),
 'ci95': [float(x) for x in np.quantile(boot, [.025, .975])],
 'limitations': 'Bootstrap de jours indépendants supposés ; la dépendance entre jours et la composition des groupes persistent.'}
for name, result in REPORT['tests'].items():
 print(name, json.dumps(result, ensure_ascii=False, indent=2))
print('Sensibilité par jours :', REPORT['day_bootstrap'])
print('Une p-value égale à 0 dans le calcul signifie un sous-flux numérique, pas une probabilité exactement nulle.')

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


mann_whitney {
  "h0": "Distributions matin et soir identiques",
  "alternative": "two-sided",
  "n_morning": 30945,
  "n_evening": 17399,
  "statistic_U_evening": 328719527.5,
  "pvalue": 0.0,
  "pvalue_holm": 0.0,
  "cliffs_delta_evening_vs_morning": 0.22107045875858033,
  "median_morning": -3.0,
  "median_evening": 0.0,
  "reject_h0_nominal": true
}
kruskal_wallis {
  "h0": "Distributions de rangs des quatre mois identiques",
  "statistic_H": 421.77319381105144,
  "pvalue": 4.2522643486064904e-91,
  "pvalue_holm": 4.2522643486064904e-91,
  "epsilon_squared": 0.005234926668971592,
  "monthly_medians": {
    "1": -1.0,
    "4": -2.0,
    "7": -1.0,
    "11": -2.0
  },
  "reject_h0_nominal": true
}
Sensibilité par jours : {'days': 122, 'replicates': 5000, 'mean_daily_evening_minus_morning_minutes': 8.662615228736968, 'ci95': [7.441150594241894, 9.913604160852813], 'limitations': 'Bootstrap de jours indépendants supposés ; la dépendance entre jours et la composition des groupes persiste

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [3]:
GOLD_SEGMENTS_PATH = '/lakehouse/gold/airport_segments'
# Agrégation distribuée : seuls 322 aéroports au maximum sont rapatriés vers le driver.
entities = (silver.groupBy('origin_airport', 'origin_airport_name', 'origin_city', 'origin_state',
                            'origin_latitude', 'origin_longitude')
 .agg(F.count('*').alias('flights'), F.count('departure_delay_15').alias('observed_flights'),
      F.sum('cancelled').alias('cancelled_flights'), F.sum('departure_delay_15').alias('delayed_15_flights'),
      F.sum('positive_departure_delay').alias('total_delay_minutes'))
 .withColumn('delay_rate_15', F.expr('try_divide(delayed_15_flights, observed_flights)'))
 .withColumn('cancellation_rate', F.expr('try_divide(cancelled_flights, flights)'))
 .withColumn('mean_positive_delay', F.expr('try_divide(total_delay_minutes, observed_flights)'))
 .orderBy('origin_airport'))
entities_pd = entities.toPandas()
assert entities_pd.origin_airport.is_unique and int(entities_pd.flights.sum()) == 1943809
FEATURES = ['flights', 'delay_rate_15', 'mean_positive_delay', 'cancellation_rate']
MIN_OBSERVED = 100  # Écarter de l'entraînement les taux fondés sur trop peu de vols.
eligible = (entities_pd.observed_flights >= MIN_OBSERVED) & entities_pd[FEATURES].notna().all(axis=1)
train = entities_pd.loc[eligible, FEATURES].astype(float).copy()
assert (train >= 0).all().all()
train['flights'] = np.log1p(train['flights'])
train['mean_positive_delay'] = np.log1p(train['mean_positive_delay'])
scaler = StandardScaler()
X = scaler.fit_transform(train)
assert np.isfinite(X).all()
# Comparer K=2..6 ; éviter les clusters de moins de 10 aéroports dans ce petit référentiel.
candidates = []
models = {}
for k in range(2, 7):
 model = KMeans(n_clusters=k, n_init=30, random_state=42).fit(X)
 sizes = np.bincount(model.labels_, minlength=k)
 candidates.append({'k': k, 'silhouette': float(silhouette_score(X, model.labels_)),
                    'inertia': float(model.inertia_), 'min_cluster_size': int(sizes.min())})
 models[k] = model
allowed = [c for c in candidates if c['min_cluster_size'] >= 10]
assert allowed, 'Aucune segmentation suffisamment peuplée : revoir les paramètres.'
best = max(allowed, key=lambda c: (c['silhouette'], -c['k']))
model = models[best['k']]
entities_pd['cluster_id'] = -1
entities_pd.loc[eligible, 'cluster_id'] = model.labels_
# Labels descriptifs, déduits des médianes originales ; aucune notion de « bon/mauvais » absolue.
profiles = entities_pd.loc[eligible].groupby('cluster_id')[FEATURES].median().sort_values('flights')
reference = entities_pd.loc[eligible, FEATURES].median()
labels = {}
for rank, (cluster, values) in enumerate(profiles.iterrows(), 1):
 volume = 'trafic élevé' if values['flights'] >= reference['flights'] else 'trafic modeste'
 delays = 'retards fréquents' if values['delay_rate_15'] >= reference['delay_rate_15'] else 'retards moins fréquents'
 labels[int(cluster)] = f'Profil {rank} — {volume}, {delays}'
entities_pd['segment'] = entities_pd.cluster_id.map(labels).fillna('Données limitées — moins de 100 vols observés')
assert entities_pd.segment.notna().all()
REPORT['clustering'] = {'features': FEATURES, 'log1p_features': ['flights', 'mean_positive_delay'],
 'minimum_observed_flights': MIN_OBSERVED, 'eligible_airports': int(eligible.sum()),
 'excluded_airports': int((~eligible).sum()), 'candidates': candidates, 'selected_k': best['k'],
 'silhouette': best['silhouette'], 'labels': labels,
 'scaler_mean': scaler.mean_.tolist(), 'scaler_scale': scaler.scale_.tolist(),
 'centers_standardized': model.cluster_centers_.tolist(),
 'profiles': [{**{'cluster_id': int(cluster), 'segment': labels[int(cluster)],
                 'airports': int((entities_pd.cluster_id == cluster).sum())},
               **{c: float(row[c]) for c in FEATURES}} for cluster, row in profiles.iterrows()]}
print(pd.DataFrame(candidates).to_string(index=False))
print('K retenu :', best['k'], '| silhouette :', best['silhouette'])
print(entities_pd.groupby('segment')[FEATURES].median().to_string())
# Conversion Spark explicite : les coordonnées absentes restent NULL, jamais NaN sur la carte.
records = entities_pd.astype(object).where(pd.notna(entities_pd), None).to_dict('records')
segments = spark.createDataFrame(records)
segments.coalesce(1).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(GOLD_SEGMENTS_PATH)
check = spark.read.format('delta').load(GOLD_SEGMENTS_PATH)
assert check.count() == 322 and check.select('origin_airport').distinct().count() == 322
assert check.agg(F.sum('flights')).first()[0] == 1943809
assert check.filter('segment IS NULL').count() == 0
REPORT['segments'] = {'path': GOLD_SEGMENTS_PATH, 'rows': check.count(), 'total_flights': 1943809}
ROOT.joinpath('reports/j4_first_half.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
print('Tests et segmentation exécutés ; dashboard laissé à réaliser.')

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


 k  silhouette    inertia  min_cluster_size
 2    0.272898 884.663510               108
 3    0.297963 660.739728                74
 4    0.293963 524.013762                43
 5    0.295481 443.724259                29
 6    0.260221 393.346878                18
K retenu : 3 | silhouette : 0.29796288337476645
                                                    flights  delay_rate_15  mean_positive_delay  cancellation_rate
segment                                                                                                           
Données limitées — moins de 100 vols observés          52.0       0.246825            15.007998           0.010556
Profil 1 — trafic modeste, retards fréquents          348.5       0.152747            12.173639           0.035834
Profil 2 — trafic modeste, retards moins fréquents    622.5       0.104736             8.241240           0.010133
Profil 3 — trafic élevé, retards fréquents           3368.0       0.160109            11.538023           0.01328

Tests et segmentation exécutés ; dashboard laissé à réaliser.


## Résultats — tests et segmentation

- Échantillon déterministe : **80,000 vols** parmi **1,917,902** vols non annulés dont le retard est renseigné.
- Mann–Whitney : matin **30,945**, soir **17,399**. Médianes échantillonnées **-3 min** et **0 min**, delta de Cliff soir/matin **0.221**. H0 rejetée nominalement après Holm ; la p-value calculée est zéro par sous-flux numérique, ce qui ne signifie pas une probabilité exactement nulle. Le test compare les distributions, pas seulement leurs médianes.
- Kruskal–Wallis : **H = 421.77**, **p corrigée = 4.25e-91**, **epsilon² = 0.0052**. L'effet mensuel de rang est faible malgré la très petite p-value. Le test global ne désigne pas les paires de mois différentes.
- Sensibilité par jours : écart moyen des moyennes journalières soir − matin **8.66 min**, intervalle bootstrap 95 % **[7.44 ; 9.91]**, sur **122 jours**. Chaque jour a ici le même poids ; ce n'est pas l'écart de moyennes pondéré par le nombre de vols. L'indépendance entre jours reste une hypothèse.
- K-Means : **K = 3**, silhouette **0.298**, **304 aéroports entraînés**, **18 hors entraînement faute de 100 vols observés. Les **322 aéroports** et **1 943 809 vols** restent représentés dans Gold.

La séparation des groupes est modérée. K=3 n'est que légèrement devant K=5 : les clusters sont une aide descriptive, pas une preuve de trois catégories naturelles. Les mots « élevé » et « fréquent » sont relatifs à la médiane des aéroports éligibles, et non à une norme de ponctualité. Les échantillons de vols partagent des dates et des perturbations ; les résultats ne prouvent pas que l'heure ou le mois cause le retard.

**Suite volontairement laissée à Amine : le dashboard Streamlit (section 3).** Voir `DASHBOARD-A-FAIRE.md`.


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [4]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")
assert "airport_segments" in gold_tables
spark.stop()

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/airline_monthly_performance :        54 lignes


 - /lakehouse/gold/airport_delay_pareto      :       322 lignes


 - /lakehouse/gold/airport_segments          :       322 lignes


 - /lakehouse/gold/daily_delay_trend         :       122 lignes


 - /lakehouse/gold/hourly_delay_profile      :        96 lignes
